# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR<sup>2</sup> dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library and the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset is loaded by its URL Croissant schema, which fully describes the dataset content, file structure, and field semantics.

In [ ]:
# Install the mlcroissant library if needed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL to the Croissant JSON-LD schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load metadata
dataset = mlc.Dataset(croissant_url)
# Display basic metadata from schema
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, their `@id` fields, and associated fields. The Croissant schema organizes dataset tables as **record sets**, each identified by an `@id` (e.g., `'cr:recordSet/0'`).

> **Note**: All entities are referenced by their full Croissant `@id` strings, not just their readable name.

In [ ]:
# Inspect record sets available in the metadata
df_rs = dataset.metadata.recordSets
print(f"Total record sets: {len(df_rs)}\n")
for rs in df_rs:
    print(f"@id: {rs.id if hasattr(rs, 'id') else rs['@id']}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields and columns for this record set
    if hasattr(rs, 'fields'):
        print(f"  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id}")
    if hasattr(rs, 'columns'):
        print(f"  Columns (by @id): {[c.id for c in rs.columns]}")
    print()

## 3. Data Extraction
Load data from a selected record set into a Pandas DataFrame. Here, we pick the main record set (usually the first or most complete), referencing via its `@id`.

**Tip**: Use only the `@id` string for referencing record sets.

In [ ]:
# Build a list of all record set @id(s)
record_sets = [r.id for r in dataset.metadata.recordSets]
dataframes = {}
for rs_id in record_sets:
    # Load all records from this record set
    recs = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(recs)

# Display column names and preview main table
main_rs_id = record_sets[0]  # Replace with specific @id if preferred
print(f"Main record set @id: {main_rs_id}")
print("Columns:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process numerical columns: filter, normalize, and group by a categorical field.

Choose a **numeric field** and a **group field** from the DataFrame. Replace the examples below with field `@id` strings from your output above.

In [ ]:
# Example numeric field: 'cr:field/6' (replace this with an actual numeric field @id from above)
# Example group field: 'cr:field/2' (replace this accordingly)
numeric_field = None
for col in dataframes[main_rs_id].columns:
    if dataframes[main_rs_id][col].dtype in [float, int] or dataframes[main_rs_id][col].dtype.name.startswith("float"):
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = dataframes[main_rs_id].select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: {numeric_field}")

# Example: Filter values greater than threshold (for demo threshold=10)
threshold = 10
df = dataframes[main_rs_id]
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold}:\n", filtered_df.head())

# Normalize the numeric column
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by a candidate categorical field if available
# Pick the first object or string-typed field excluding the numeric field
group_field = None
for col in df.columns:
    if df[col].dtype == object and col != numeric_field:
        group_field = col
        break
if group_field:  # Only group if a suitable field exists
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped {numeric_field} mean by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields. We'll plot the distribution of the numeric field and (optionally) the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=25, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by the group field if available
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"Distribution of {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the [FAIR² mass spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using a Croissant schema and the `mlcroissant` library, previewed metadata and schema structure, extracted all record sets by their Croissant `@id`, performed simple data filtering and normalization, grouped and visualized records.

- All references to tables/fields use full Croissant `@id`s.
- For deeper exploration, review the metadata fields and schema [here](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

**Key findings and next steps:**
- The available record sets reflect tables for protein mass spectrometry quantification and annotations.
- Data can be filtered and grouped by numeric and categorical properties, as illustrated above.
- For scientific and research analysis, further domain-specific filters and visualizations can be developed using the Croissant `@id` mappings.